In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, gc, os, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_score, recall_score, accuracy_score)

os.makedirs('../Models', exist_ok=True)
print("✅ Imports OK !")
print(f"   Pandas : {pd.__version__} | NumPy : {np.__version__}")

# Fix renderer
import sys
sys.stdout.flush()

In [4]:
print("📂 Chargement 6.3M lignes...")
df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")
print(f"✅ {df.shape[0]:,} lignes chargées")

# Feature Engineering
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
df['isZeroOrig'] = (df['oldbalanceOrg'] == 0).astype(int)
df['isZeroDest']  = (df['newbalanceDest'] == 0).astype(int)
df['amountRatio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
tx_per_step = df.groupby('step')['amount'].count().reset_index()
tx_per_step.columns = ['step', 'nb_tx_step']
df = df.merge(tx_per_step, on='step', how='left')
df['avg_amount_by_type'] = df.groupby('type_encoded')['amount'].transform('mean')
df['is_large_tx'] = (df['amount'] > df['avg_amount_by_type'] * 2).astype(int)

FEATURES = [
    'type_encoded', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig',
    'errorBalanceDest', 'isZeroOrig', 'isZeroDest',
    'amountRatio', 'nb_tx_step', 'is_large_tx'
]

X = df[FEATURES].astype('float32')
y = df['isFraud']

del df; gc.collect()

# Vérification
print(f"✅ 13 features créées — RAM libérée")
print(f"   Shape X : {X.shape}")
print(f"   Fraudes : {y.sum():,} ({y.mean()*100:.3f}%)")
print(f"   X dtype : {X.dtypes[0]}")
import sys; sys.stdout.flush()

📂 Chargement 6.3M lignes...
✅ 6,362,620 lignes chargées
✅ 13 features créées — RAM libérée
   Shape X : (6362620, 13)
   Fraudes : 8,213 (0.129%)
   X dtype : float32
